# OmniLingual 300M Levantine Full Non-Smoke Notebook

This is a dedicated copy of the plug-and-play notebook configured for `facebook/omniASR-LLM-300M` on the real non-smoke Levantine dataset. It is wired to materialize the full selected corpus locally, then run the plug-and-play `prepare` export flow. For actual OmniLingual fine-tuning, use the official external Omni recipe after this prepare step.

## Connected data

This notebook is connected to the full Levantine selection used by the existing OmniLingual run:

- Train: 1,859 rows total
- Train mix: 1,514 rows from `casablanca_levantine_train` and 345 rows from `omnilingual_apc_north_levantine_train`
- Validation: 843 rows total
- Validation mix: 757 rows from `casablanca_levantine_eval_pool` and 86 rows from `omnilingual_apc_north_levantine_eval_pool`
- Test: 843 rows total
- Test mix: 757 rows from `casablanca_levantine_eval_pool` and 86 rows from `omnilingual_apc_north_levantine_eval_pool`

This is non-smoke mode: it does not use the tiny synthetic dataset.

In [ ]:
from pathlib import Path
import subprocess, sys

PROJECT_DIR = Path.cwd()
MATERIALIZE_SCRIPT = PROJECT_DIR / "prepare_omnilingual_300m_levantine_full.py"
TRAINER_SCRIPT = PROJECT_DIR / "asr_universal_trainer.py"
CONFIG_PATH = PROJECT_DIR / "omnilingual_300m_levantine_full.yaml"
WORK_DIR = PROJECT_DIR / "runs" / "omnilingual_300m_levantine_full"
DATASET_DIR = PROJECT_DIR / "datasets" / "omnilingual_300m_levantine_full"

print("Project dir:", PROJECT_DIR)
print("Materialize script:", MATERIALIZE_SCRIPT)
print("Trainer:", TRAINER_SCRIPT)
print("Config:", CONFIG_PATH)
print("Dataset dir:", DATASET_DIR)
print("Work dir:", WORK_DIR)


## 1. Materialize the full non-smoke dataset

This step converts the real selected Levantine manifests into local audio files plus JSONL metadata that the plug-and-play trainer can consume directly.

In [ ]:
cmd = [sys.executable, str(MATERIALIZE_SCRIPT)]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, result.returncode


In [ ]:
for rel in ["dataset_summary.json", "train.jsonl", "validation.jsonl", "test.jsonl"]:
    path = DATASET_DIR / rel
    print(f"\n--- {path} ---")
    if path.exists():
        text = path.read_text(encoding="utf-8")
        print(text[:4000])
    else:
        print("missing")


## 2. Show the dedicated non-smoke config

In [ ]:
print(CONFIG_PATH.read_text(encoding="utf-8"))


## 3. Run the plug-and-play prepare export

For OmniLingual 300M in this trainer, the verified supported phase is `prepare`. This writes the normalized train/validation/test JSONL files and export artifacts for the external Omni recipe.

In [ ]:
cmd = [sys.executable, str(TRAINER_SCRIPT), "--config", str(CONFIG_PATH), "--stage", "prepare"]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, result.returncode


## 4. Inspect outputs

In [ ]:
for rel in [
    "run.log",
    "run_result.json",
    "prepared/stats.json",
    "prepared/dataset_resolved.json",
    "prepared/model_spec.json",
]:
    path = WORK_DIR / rel
    print(f"\n--- {path} ---")
    if path.exists():
        print(path.read_text(encoding="utf-8")[:5000])
    else:
        print("missing")


## 5. Next step for real OmniLingual training

This notebook copy is set up in non-smoke mode on the full real dataset. The next real training step is to use the exported dataset with the official OmniLingual external recipe rather than expecting the generic HF trainer path to fine-tune Omni models directly.